In [1]:
%load_ext autoreload
%autoreload 2

In [4]:
from fipilot.resume_extraction import ResumeExtract
from fipilot.configs.settings import cfg
extractor = ResumeExtract(
    yolo_model=cfg.YOLO_MODEL,
    dpi=150,
)
extractor.pipeline("/home/hoai/user/resource/fipilot/backend/test/4237bce0-2c0b-4af2-adfc-f234430fffaa.pdf")

RuntimeError: Direct model not loaded. Check 'direct_model_name' in config.yaml and ensure transformers/torch are installed.

In [ ]:
from pathlib import Path
INPUT_DATA_DIR = Path("/home/hoai/user/resource/fipilot/backend/test/")

In [2]:
input = "[0]: NGUYEN VAN DAT [1]: AI ENGINEER [2]: Dong Tam, Hai Ba Trung , Ha Noi [3]: # [4]: Hanoi University of Science and Technology [5]: Aug 2017 – Aug 2022 [6]: Engineer of Electrics and Telecommunications [7]: GPA: 3.21/4.0 [8]: Technical Skills [9]: - TOEIC: 640 [10]: - Programming Languages: Python, C++ [11]: - Developer Tools: VS Code, Pycharm [12]: - Technologies: K8s, Kafka, GitHub, Docker, GitLab CI/CD, Stable Diffusion, LLM, Langchain (RAG) [13]: - Framework: Triton, Pytorch, Tensorflow, FastAPI, ONNX, Keras [14]: - Database/Storge: MySQL, MongoDB, S3 [15]: Experience [16]: HBLab [17]: Apr 2024 - Now [18]: AI Engineer [19]: Ha Noi [20]: - Participating in the development Japanese POCR system (Extract Layout, Table Reconstruction) [21]: - Maintaining and developing functions for the Japanese OCR system (C++, Java) [22]: - Researching pipeline LLM using Langchain(RAG) [23]: Onsite MB Bank [24]: Mar 2022 - Mar 2024 [25]: AI Engineer [26]: Ha Noi [27]: - Developed a solution and pipeline full pipeline OCR for Bank. Pipeline: Text Detection, Rotate Image, Text [28]: Recognition, Extraction Information. Optimize: Accuracy about 94% - 95% per paper, and Performance about 1s - [29]: 3s per paper on triton [30]: - Participating in the backend API development of the MLOps MB Bank. Automated Model Deployment and [31]: Training Functionality (Kafka, MongoDB) [32]: - Developing and Building an eKYC Solution for Banking. Accuracy OCR: 98% , developing a model classification id [33]: card(21 classes) with accuracy [34]: 99% [35]: - Building and Developing a full pipeline signature verification. Pipeline: Model signature detection, Model the role of [36]: the signer detection to find out if the person signed or not [37]: - Developing a Feature Extraction Model for Image Repository in a Bank using EfficientNet Model. [38]: Viettel Post [39]: Jan 2021 – Mar 2022 [40]: Intern - AI Engineer [41]: Ha Noi [42]: - Developing an Object Detection Model for Viettel Post Warehouse. Programming C to control servo (a leaser distance [43]: sensor be assembled with a servo), return a distance from laser to object. Programming python to segmentation object [44]: with U2Net model, return area of the object. Return a result of estimate the volume. Volume = Area of the object [45]: Height [46]: Lab EDABK [47]: May 2020 – Aug 2022 [48]: Research Student [49]: HUST [50]: - Building and developing a model for detecting defects in phone screen products, deploying it on a server. [51]: - Building and developing a model to address the problem of detecting and tracking intruders in a monitored area [52]: using the DeepStream platform. [53]: Prizes and Awards [54]: Hackathon [55]: Jun 2021 [56]: Archive 2nd [57]: Hackathon [58]: - Won Second Award: International Hackathon IT Solutions for Business held by Irkutsk National Research Technical [59]: University on June 2-7, 2021 Masters Scholarship at Irkutsk National Research Technical University [60]: HUST [61]: Aug 2020 [62]: Annual Incentive Scholarship of HUST University (20201)"
result = {
  "workExperience": [
    {
      "companyName": "HBLab",
      "position": "AI Engineer",
      "employmentPeriod": {
        "startDate": "Apr 2024",
        "endDate": "Now"
      },
      "jobDescription_refer_index_range": [
        18,
        22
      ],
      "internship": 0
    },
    {
      "companyName": "Onsite MB Bank",
      "position": "AI Engineer",
      "employmentPeriod": {
        "startDate": "Mar 2022",
        "endDate": "Mar 2024"
      },
      "jobDescription_refer_index_range": [
        25,
        31
      ],
      "internship": 0
    },
    {
      "companyName": "Viettel Post",
      "position": "Intern - AI Engineer",
      "employmentPeriod": {
        "startDate": "Jan 2021",
        "endDate": "Mar 2022"
      },
      "jobDescription_refer_index_range": [
        40,
        46
      ],
      "internship": 1
    },
    {
      "companyName": "Lab EDABK",
      "position": "Research Student",
      "employmentPeriod": {
        "startDate": "May 2020",
        "endDate": "Aug 2022"
      },
      "jobDescription_refer_index_range": [
        47,
        52
      ],
      "internship": 0
    }
  ],
  "education": [
    {
      "school": "Hanoi University of Science and Technology",
      "major": "Engineer of Electrics and Telecommunications",
      "degreeLevel": "Bachelor's",
      "period": {
        "startDate": "2017.08",
        "endDate": "2022.08"
      },
      "educationDescription": "GPA: 3.21/4.0"
    }
  ]
}

In [3]:
import re

def parse_indexed_text(input_str):
    """Tách input dạng '[0]: text [1]: text ...' thành dict {index: text}"""
    pattern = re.compile(r'\[(\d+)\]:\s*(.*?)(?=\[\d+\]:|$)', re.S)
    return {int(idx): text.strip() for idx, text in pattern.findall(input_str)}

def range_to_text(index_map, start, end):
    """Ghép text từ index start->end (bao gồm cả 2 đầu) thành 1 đoạn"""
    return " ".join(index_map[i] for i in range(start, end + 1) if i in index_map)

def replace_refer_index_range(data, input_str,
                               key_name="jobDescription_refer_index_range",
                               new_key="jobDescription"):
    """
    Duyệt đệ quy qua dict/list, tìm key jobDescription_refer_index_range,
    thay bằng text tương ứng lấy từ input theo range, gán vào key mới jobDescription.
    """
    index_map = parse_indexed_text(input_str)

    def _walk(node):
        if isinstance(node, dict):
            if key_name in node:
                start, end = node[key_name]
                node[new_key] = range_to_text(index_map, start, end)
                del node[key_name]
            for v in node.values():
                _walk(v)
        elif isinstance(node, list):
            for item in node:
                _walk(item)

    _walk(data)
    return data
output = replace_refer_index_range(result, input)
output

{'workExperience': [{'companyName': 'HBLab',
   'position': 'AI Engineer',
   'employmentPeriod': {'startDate': 'Apr 2024', 'endDate': 'Now'},
   'internship': 0,
   'jobDescription': 'AI Engineer Ha Noi - Participating in the development Japanese POCR system (Extract Layout, Table Reconstruction) - Maintaining and developing functions for the Japanese OCR system (C++, Java) - Researching pipeline LLM using Langchain(RAG)'},
  {'companyName': 'Onsite MB Bank',
   'position': 'AI Engineer',
   'employmentPeriod': {'startDate': 'Mar 2022', 'endDate': 'Mar 2024'},
   'internship': 0,
   'jobDescription': 'AI Engineer Ha Noi - Developed a solution and pipeline full pipeline OCR for Bank. Pipeline: Text Detection, Rotate Image, Text Recognition, Extraction Information. Optimize: Accuracy about 94% - 95% per paper, and Performance about 1s - 3s per paper on triton - Participating in the backend API development of the MLOps MB Bank. Automated Model Deployment and Training Functionality (Kafka